# Stage 4A — Boosting Infrastructure and Experiment Foundation

This notebook builds safe shared tools for later CatBoost, LightGBM, and XGBoost work. It does not fit a real boosting model. The saved Test Set stays locked.

## 0. Stage Objective

The goal is to prepare one safe and reusable foundation. This includes fixed training-only samples, shared metrics, Registry rules, atomic file writes, worker timeouts, progress files, and environment evidence.

In [1]:
from datetime import datetime, timezone
run_id = datetime.now(timezone.utc).isoformat()
print("Stage 4A foundation only")
print("Real boosting fits allowed:", 0)
print("Test predictions allowed:", 0)

Stage 4A foundation only
Real boosting fits allowed: 0
Test predictions allowed: 0


## 1. Imports and Configuration

The random seed is 42. New internal identifiers use `stage4a`. Sample roles and target bins are metadata only. They are never model features.

In [2]:
import json
import sys
from pathlib import Path
import pandas as pd
import stage4_boosting_utils as s4

SEED = s4.RANDOM_SEED
STAGE_ID = s4.STAGE_ID
print({"stage_id": STAGE_ID, "seed": SEED, "python": sys.version.split()[0]})

{'stage_id': 'stage4a', 'seed': 42, 'python': '3.12.3'}


## 2. Project Discovery

The project root is found from `AGENTS.md` and the saved Train row file. This avoids depending on one fixed working directory.

In [3]:
ROOT = s4.discover_project_root()
print("Project root:", ROOT)
print("Source CSV files:", sorted(path.name for path in (ROOT / "data").glob("*.csv")))

Project root: D:\SHARIF\TERM7\DATA\PROJECT\regresionpart2
Source CSV files: ['regression_with_sensitive_features.csv', 'regression_without_sensitive_features.csv']


## 3. Previous-Stage Validation

Stage 1, Stage 2, and Stage 3 must report PASS. The saved Train, Test, and Fold files must be complete and separate. No replacement split is allowed.

In [4]:
previous = json.loads((ROOT / "artifacts/reports/stage4a_previous_stage_validation.json").read_text(encoding="utf-8"))
assert previous["status"] == "PASS", previous
previous

{'reports': {'prompt1_verification.json': {'exists': True,
   'status': 'PASS',
   'sha256': '50172f0f3767f2326f3ab626f25bd483f69c7f625ee847d82f4e581495aeaafb'},
  'prompt2_verification.json': {'exists': True,
   'status': 'PASS',
   'sha256': '38be7bec7cb1b44227b0a5ebfcd7b099b1905d4bd92527eb9e237a78c8c530eb'},
  'stage3_verification.json': {'exists': True,
   'status': 'PASS',
   'sha256': '119b86b0d6119c9e4359c1acf4dfac69d6882e5df9ee744b928db43fe4a72e20'}},
 'train_rows': 399788,
 'locked_test_rows': 99948,
 'fold_rows': 399788,
 'fold_values': [0, 1, 2],
 'split_valid': True,
 'test_targets_loaded': False,
 'test_predictions_created': False,
 'status': 'PASS'}

## 4. Protected File Manifest

The before-run manifest includes source data, all prior notebooks, saved splits, prior results, prior models, and prior predictions. Any changed hash is a critical failure.

In [5]:
before_path = ROOT / "artifacts/manifests/stage4/stage4a_protected_hashes_before.json"
before = json.loads(before_path.read_text(encoding="utf-8"))
protected_check = s4.recheck_protected_manifest(ROOT, before)
s4.atomic_write_json(ROOT / "artifacts/manifests/stage4/stage4a_protected_hashes_after.json", protected_check)
assert protected_check["status"] == "PASS", protected_check["mismatches"]
print({"protected_files": protected_check["file_count"], "mismatches": len(protected_check["mismatches"])})

{'protected_files': 266, 'mismatches': 0}


## 5. Environment and Package Audit

The audit records Python, packages, CPU, RAM, disk, GPU, and CUDA. Missing authorized packages received at most one project-local installation attempt before notebook execution. This section imports and constructs small estimator objects only. It does not call `fit`.

In [6]:
environment = json.loads((ROOT / "artifacts/reports/stage4a_environment.json").read_text(encoding="utf-8"))
worker_package_smoke = json.loads((ROOT / "artifacts/reports/stage4a_clean_worker_package_smoke_test.json").read_text(encoding="utf-8"))
assert worker_package_smoke["status"] == "PASS"
package_view = {
    name: {
        "version": item.get("module_version") or item.get("global_version"),
        "import_ok": item.get("import_ok", item.get("global_version") is not None),
        "construction_ok": item.get("construction_ok"),
    }
    for name, item in environment["packages"].items()
}
pd.DataFrame(package_view).T

,version,import_ok,construction_ok
pandas,2.2.2,True,None
numpy,2.2.6,True,None
scikit_learn,1.9.0,True,None
joblib,1.5.3,True,None
catboost,1.2.10,True,True
lightgbm,4.6.0,True,True
xgboost,3.3.0,True,True
shap,0.52.0,True,None


## 6. Artifact Directory Design

Each boosting library has separate result, model, and prediction folders. Shared Stage 4 reports, features, figures, manifests, and checkpoints also have separate folders. Old artifacts are not moved.

In [7]:
stage4_directories = s4.ensure_stage4_directories(ROOT)
assert all((ROOT / path).is_dir() for path in stage4_directories)
pd.DataFrame({"stage4_directory": stage4_directories})

,stage4_directory
0,artifacts/results/stage4
1,artifacts/results/stage4/catboost
2,artifacts/results/stage4/lightgbm
3,artifacts/results/stage4/xgboost
4,artifacts/models/catboost
5,artifacts/models/lightgbm
6,artifacts/models/xgboost
7,artifacts/predictions/catboost
8,artifacts/predictions/lightgbm
9,artifacts/predictions/xgboost


## 7. Shared Metric System

The shared adapter supports raw and `log1p` targets. Predictions are changed back to the original target scale before metrics are calculated.

In [8]:
metric_smoke = json.loads((ROOT / "artifacts/reports/stage4a_metric_smoke_test.json").read_text(encoding="utf-8"))
assert metric_smoke["status"] == "PASS"
pd.DataFrame([metric_smoke["raw"], metric_smoke["log1p"]], index=["raw", "log1p"])

,mae,mse,rmse,mape_percent,r_squared,rmsle,rmsle_clipped_zero,median_absolute_error,wape_percent,mean_signed_error,p90_absolute_error,negative_prediction_rate,mae_usd,rmse_usd,original_scale
raw,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.0,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,True
log1p,7.771561e-16,1.035380e-30,1.017536e-15,1.665335e-14,1.0,0.0,0.0,6.661338e-16,2.072416e-14,3.330669e-16,1.509903e-15,0.0,7.771561e-13,1.017536e-12,True


## 8. Shared Registry System

Experiment IDs are deterministic and start with `stage4a`. Upsert uses the fixed 31-column schema. The smoke test uses a scratch copy, so all 215 prior Registry rows stay unchanged.

In [9]:
registry_smoke = json.loads((ROOT / "artifacts/reports/stage4a_registry_smoke_test.json").read_text(encoding="utf-8"))
assert registry_smoke["status"] == "PASS"
registry_smoke

{'status': 'PASS',
 'main_registry_rows': 215,
 'smoke_registry_rows': 216,
 'main_registry_unchanged': True,
 'idempotent_upsert': True,
 'experiment_id': 'stage4a__adapter_smoke__without_sensitive__raw__foundation_smoke__fold-na__cfg-1ceb1cd2ffee',
 'stage_parameterized': True,
 'provenance_changes_id': True,
 'smoke_path': 'artifacts\\checkpoints\\stage4\\stage4a_registry_smoke.csv'}

## 9. Atomic Artifact Writers

CSV, JSON, and joblib files are written to a temporary file first. The complete temporary file then replaces the destination.

In [10]:
atomic_smoke = json.loads((ROOT / "artifacts/reports/stage4a_atomic_write_smoke_test.json").read_text(encoding="utf-8"))
assert atomic_smoke["status"] == "PASS"
cache_smoke = json.loads((ROOT / "artifacts/reports/stage4a_cache_validation_smoke_test.json").read_text(encoding="utf-8"))
assert cache_smoke["status"] == "PASS"
{"atomic_write": atomic_smoke, "cache_validation": cache_smoke}

{'atomic_write': {'status': 'PASS',
  'paths': ['artifacts\\checkpoints\\stage4\\atomic_smoke\\sample.csv',
   'artifacts\\checkpoints\\stage4\\atomic_smoke\\sample.json',
   'artifacts\\checkpoints\\stage4\\atomic_smoke\\sample.joblib']},
 'cache_validation': {'status': 'PASS',
  'complete_provenance_cache_accepted': True,
  'wrong_split_provenance_rejected': True,
  'wrong_file_hash_rejected': True,
  'csv_schema_rows_unique_finite_and_provenance_accepted': True,
  'csv_wrong_feature_provenance_rejected': True,
  'valid_reason': 'valid',
  'wrong_split_reason': 'metadata_mismatch:split_digest',
  'wrong_hash_reason': 'sha256_mismatch',
  'csv_valid_reason': 'valid',
  'csv_wrong_feature_reason': 'metadata_mismatch:feature_digest',
  'path': 'artifacts\\checkpoints\\stage4\\cache_validation_smoke.json',
  'csv_path': 'artifacts\\checkpoints\\stage4\\cache_validation_smoke.csv'}}

## 10. Worker and Timeout Design

A parent process starts each later heavy worker. If time expires, the parent stops the worker and its child process tree. A check after `fit` is not used as the timeout.

In [11]:
timeout_smoke = json.loads((ROOT / "artifacts/reports/stage4a_timeout_smoke_test.json").read_text(encoding="utf-8"))
assert timeout_smoke["pass"], timeout_smoke
timeout_smoke

{'status': 'timed_out',
 'timed_out': True,
 'return_code': 15,
 'wall_seconds': 1.062999999994645,
 'stdout': '',
 'stderr': '',
 'cleanup': {'pid': 3480,
  'method': 'psutil_recursive_tree',
  'return_code': None,
  'descendant_pids': [2888],
  'alive_after_cleanup': [],
  'worker_return_code': 15},
 'heartbeat_created': True,
 'worker_process_stopped': True,
 'child_process_id': 2888,
 'child_process_alive_after_cleanup': False,
 'pass': True}

## 11. Progress and Heartbeat System

Progress files describe the current Stage step. Heartbeat files show that a worker is alive. Both use atomic JSON writes.

In [12]:
smoke_suite = json.loads((ROOT / "artifacts/reports/stage4a_smoke_suite.json").read_text(encoding="utf-8"))
assert smoke_suite["status"] == "PASS"
heartbeat_path = ROOT / "artifacts/checkpoints/stage4/stage4a_notebook_heartbeat.json"
s4.write_heartbeat(heartbeat_path, "stage4a_notebook", state="foundation_ready")
heartbeat = json.loads(heartbeat_path.read_text(encoding="utf-8"))
print({"smoke_suite": smoke_suite["status"], "heartbeat_state": heartbeat["state"]})

{'smoke_suite': 'PASS', 'heartbeat_state': 'foundation_ready'}


## 12. Discovery Sample

The discovery sample has 50,000 training rows and 15,000 validation rows. It is for first models, first feature importance, and first error analysis in Stage 4B.

In [13]:
sample_verification = s4.validate_existing_stage4_samples(ROOT)
assert sample_verification["status"] == "PASS"
discovery = pd.read_csv(ROOT / "artifacts/splits/stage4/stage4_discovery_sample.csv")
discovery.groupby(["sample_role", "target_bin"]).size().unstack(fill_value=0)

target_bin,0,1,2,3,4,5,6,7,8,9
sample_role,,,,,,,,,,
train,5113,4948,4978,5050,4948,5070,4909,5045,4945,4994
validation,1534,1485,1493,1515,1485,1521,1473,1513,1483,1498


## 13. Feature Confirmation Sample

The feature confirmation sample has 80,000 training rows and 20,000 validation rows. It checks that a later feature idea also helps on different rows.

In [14]:
confirmation = pd.read_csv(ROOT / "artifacts/splits/stage4/stage4_feature_confirmation_sample.csv")
assert len(confirmation) == 100_000 and confirmation["row_id"].is_unique
confirmation.groupby(["sample_role", "target_bin"]).size().unstack(fill_value=0)

target_bin,0,1,2,3,4,5,6,7,8,9
sample_role,,,,,,,,,,
train,8181,7917,7965,8080,7917,8112,7855,8072,7911,7990
validation,2045,1979,1991,2020,1979,2028,1964,2018,1978,1998


## 14. Final Selection Sample

The final selection sample has 100,000 training rows and 25,000 validation rows. It is reserved for limited final tuning and the later sensitive comparison.

In [15]:
final_selection = pd.read_csv(ROOT / "artifacts/splits/stage4/stage4_final_selection_sample.csv")
assert len(final_selection) == 125_000 and final_selection["row_id"].is_unique
final_selection.groupby(["sample_role", "target_bin"]).size().unstack(fill_value=0)

target_bin,0,1,2,3,4,5,6,7,8,9
sample_role,,,,,,,,,,
train,10226,9897,9956,10100,9897,10140,9819,10089,9889,9987
validation,2557,2474,2489,2525,2474,2535,2455,2522,2472,2497


## 15. Sample Verification

All three samples use saved Train rows only. They are disjoint, have zero Test overlap, and closely follow the saved training target-bin distribution.

In [16]:
sample_verification = json.loads((ROOT / "artifacts/splits/stage4/stage4_sample_verification.json").read_text(encoding="utf-8"))
assert sample_verification["status"] == "PASS"
pd.DataFrame(sample_verification["samples"]).T

,rows,train_rows,validation_rows,expected_rows,row_ids_unique,maximum_target_bin_proportion_difference,role_maximum_target_bin_proportion_difference,sha256,valid
discovery,65000,50000,15000,65000,True,0.000009,"{'train': 1.2069997098462792e-05, 'validation'...",79cf8051552f1edc1e68b9284d0a3ccfb49f5ceb3afd4b...,True
feature_confirmation,100000,80000,20000,100000,True,0.000012,"{'train': 7.06999709847167e-06, 'validation': ...",2dabbdc5e6b0fee9bc50e63feeb912e3096a5408cafbbf...,True
final_selection,125000,100000,25000,125000,True,0.000008,"{'train': 5.974866679345214e-06, 'validation':...",445a077b3fc1963bec38d8ba90384df2e1400e3897b25f...,True


## 16. Stage 4A Artifact Summary

The summary lists Stage 4A foundation artifacts. Model and prediction folders stay empty because no real boosting experiment is allowed here.

In [17]:
artifact_summary = s4.build_artifact_summary(ROOT)
assert artifact_summary["real_boosting_models_trained"] == 0
assert artifact_summary["test_predictions_created"] == 0
print({"artifact_count": artifact_summary["artifact_count"], "real_boosting_models": 0, "test_predictions": 0})

{'artifact_count': 264, 'real_boosting_models': 0, 'test_predictions': 0}


## 17. Stage 4A Verification

Internal verification checks prior PASS evidence, protected hashes, Test isolation, sample safety, package evidence, and all utility smoke tests. Final external verification also checks two saved notebook runs and the independent review.

In [18]:
internal_verification = s4.build_internal_verification(ROOT)
assert internal_verification["status"] == "PASS", internal_verification
pd.Series(internal_verification["checks"], name="passed")

previous_stages_pass                True
saved_split_and_folds_valid         True
test_targets_not_loaded             True
test_predictions_not_created        True
protected_hashes_unchanged          True
three_samples_valid                 True
samples_disjoint                    True
samples_have_zero_test_overlap      True
package_audit_complete              True
clean_worker_package_import_pass    True
metric_smoke_pass                   True
atomic_write_smoke_pass             True
cache_validation_smoke_pass         True
registry_smoke_pass                 True
timeout_smoke_pass                  True
main_registry_unchanged             True
no_boosting_model_fit_artifact      True
no_boosting_prediction_artifact     True
Name: passed, dtype: bool

## 18. Stage 4A Completion Note

Stage 4A is complete after final external verification. Three training-only samples were created and validated. Test data was not used. CatBoost, LightGBM, and XGBoost are available. Shared utilities are ready. No real boosting experiment was trained. The next Stage is Stage 4B.

In [19]:
execution_history = s4.record_notebook_success(ROOT, run_id)
print("Stage 4A notebook run completed successfully.")
print("Successful clean runs recorded:", execution_history["successful_run_count"])
print("Next step after final external verification: Begin Stage 4B — Initial Boosting Feature Packs.")

Stage 4A notebook run completed successfully.
Successful clean runs recorded: 5
Next step after final external verification: Begin Stage 4B — Initial Boosting Feature Packs.


## 19. Stage 4B Objective

Stage 4B creates safe initial Feature Packs for CatBoost, LightGBM, and XGBoost. It uses saved Train rows only. It does not compare models or tune settings.

In [2]:
stage4b_summary = b.build_stage4b_artifacts(ROOT)
assert stage4b_summary["status"] == "PASS"
stage4b_start = json.loads((ROOT / "artifacts/reports/stage4b_start_validation.json").read_text(encoding="utf-8"))
display(pd.DataFrame([stage4b_start["checks"]]).T.rename(columns={0: "passed"}))
display(stage4b_summary)

,passed
stage4a_verification_pass,True
stage4a_all_checks_pass,True
three_training_only_samples_pass,True
sample_test_overlap_zero,True
protected_files_unchanged,True
package_report_complete,True
shared_utility_imports,True
test_values_not_loaded,True


{'stage': 'stage4b',
 'version': 'stage4b_initial_boosting_packs_v1_20260714',
 'created_at_utc': '2026-07-14T09:52:01.999722+00:00',
 'feature_audit_rows': 35,
 'proposal_rows': 12,
 'selected_feature_rows': 8,
 'rejected_feature_rows': 4,
 'feature_packs': ['boosting_base_v1',
  'boosting_engineered_v1',
  'catboost_native_v1',
  'lightgbm_encoded_v1',
  'xgboost_sparse_v1'],
 'schemas': ['catboost', 'lightgbm', 'xgboost'],
 'transformer_roundtrip_rows': 5,
 'leakage_review_status': 'PASS',
 'smoke_status': 'PASS',
 'internal_verification_status': 'PASS',
 'test_rows_used': 0,
 'real_boosting_screening_performed': False,
 'status': 'PASS'}

## 20. Existing Feature Review

The review uses the source inventory, Stage 1 features, Stage 3 proposals, selected packs, importance, and leakage reports. Strong prior signals include applicant income, lien status, state, area income, and tract income ratio.

In [3]:
feature_audit = pd.read_csv(ROOT / "artifacts/features/stage4/feature_audit.csv")
stage3_importance = pd.read_csv(ROOT / "artifacts/features/tree/importance/stage3_feature_importance_summary.csv")
display(feature_audit[["feature_name", "source_or_engineered", "cardinality", "missing_rate", "decision"]])
display(stage3_importance.loc[stage3_importance["sensitive_mode"].eq("without_sensitive")].head(15))

,feature_name,source_or_engineered,cardinality,missing_rate,decision
0,agency_name,source,6,0.0,include_base
1,applicant_income_000s,source,950,0.0,include_base
2,applicant_income_area_group,existing_engineered,4,0.0,include_base
3,applicant_income_to_area_income,existing_engineered,17917,0.0,include_base
4,census_tract_number,source,10639,0.0,include_model_specific
5,county_code,source,168,0.0,exclude_redundant
6,county_name,source,749,0.0,include_model_specific
7,family_units_per_1000_people,existing_engineered,17839,0.0,include_base
8,has_co_applicant,existing_engineered,2,0.0,include_base
9,hud_median_family_income,source,257,0.0,include_base


,source_feature,importance,importance_std,model_name,sensitive_mode,method,sample_rows,n_repeats,associative_not_causal,is_engineered,is_sensitive
0,applicant_income_000s,47.500333,0.628628,decision_tree,without_sensitive,permutation,10000,3,True,False,False
1,hud_median_family_income,17.992551,0.473876,decision_tree,without_sensitive,permutation,10000,3,True,False,False
2,lien_status_name,17.505246,0.278296,decision_tree,without_sensitive,permutation,10000,3,True,False,False
3,state_name,14.485256,0.414591,decision_tree,without_sensitive,permutation,10000,3,True,False,False
4,tract_income_ratio,13.276793,0.545271,decision_tree,without_sensitive,permutation,10000,3,True,False,False
5,owner_occupancy_name,7.255426,0.256744,decision_tree,without_sensitive,permutation,10000,3,True,False,False
6,us_region,7.189012,0.067955,decision_tree,without_sensitive,permutation,10000,3,True,False,False
7,loan_purpose_name,6.001887,0.065338,decision_tree,without_sensitive,permutation,10000,3,True,False,False
8,agency_name,3.084715,0.169031,decision_tree,without_sensitive,permutation,10000,3,True,False,False
9,family_units_per_1000_people,1.936973,0.110125,decision_tree,without_sensitive,permutation,10000,3,True,False,False


## 21. Leakage and Redundancy Audit

The target, row ID, target bins, Fold IDs, sensitive interactions, exact duplicates, and post-outcome information are excluded. Lender and geography fields may be proxy signals, so later results must stay associative.

In [4]:
leakage_text = (ROOT / "artifacts/reports/stage4b_leakage_review.md").read_text(encoding="utf-8")
assert "Status: PASS" in leakage_text
display(Markdown(leakage_text))

# Stage 4B Leakage Review

Status: PASS

## Scope

This review covers the initial boosting Feature Packs and their fixed features. It uses only non-sensitive training metadata and prior training-only evidence. It does not use locked Test values.

## Decisions

- The target `loan_amount_000s`, row IDs, target bins, and Fold IDs are not model features.
- No selected fixed feature uses a sensitive field or the target.
- No target encoding, category target mean, SHAP feature, or post-outcome field is used.
- Learned frequency, rare-category, vocabulary, imputation, ordinal, and one-hot steps stay inside model Pipelines.
- The six compact category combinations and two numeric features are fixed row-level calculations.
- Missing indicators were not selected because the training-only audit found no missing source values.
- Stage 1 monotonic duplicates and exact Stage 3 proposals were rejected.

## Model-specific safety

- CatBoost keeps reviewed categories as native text fields. It does not receive one-hot encoded categories.
- LightGBM uses fold-fit frequency and ordinal handling inside its Pipeline.
- XGBoost uses controlled sparse one-hot output and fold-fit frequency handling for high-cardinality fields.
- Unseen categories map to `<RARE>`, ordinal `-1`, an ignored one-hot value, or frequency `0` as documented by each schema.

## Residual limitations

Lender and geography fields can act as proxies for sensitive context. They have no confirmed target leakage, but later Stage 4 experiments must compare sensitive modes and interpret these fields as associative, not causal. Random Fold category overlap can also make category effects look more stable than they are.


## 22. Initial Feature Proposals

Twelve fixed features were reviewed. Eight were selected. Four were rejected because they repeat Stage 3 work or duplicate existing information.

In [5]:
initial_feature_proposals = pd.read_csv(ROOT / "artifacts/features/stage4/initial_feature_proposals.csv")
assert len(initial_feature_proposals) <= 12
display(initial_feature_proposals)

,feature_name,formula,source_columns,expected_benefit,data_type,missing_value_behavior,zero_denominator_behavior,sensitive_derived,target_derived,leakage_status,selected,rejection_reason
0,estimated_tract_family_income_000s,(hud_median_family_income / 1000.0) * tract_in...,hud_median_family_income|tract_income_ratio,Places tract-relative family income on the app...,numeric,A missing or non-finite source gives NaN for p...,No denominator.,False,False,PASS,True,NaN
1,applicant_vs_area_income_gap_000s,applicant_income_000s - (hud_median_family_inc...,applicant_income_000s|hud_median_family_income,Adds a signed absolute income gap beside the e...,numeric,A missing or non-finite source gives NaN for p...,No denominator.,False,False,PASS,True,NaN
2,purpose_lien_status_group,loan_purpose_name + ' | ' + lien_status_name,loan_purpose_name|lien_status_name,Combines loan use with a strong lien-position ...,categorical,Each missing part becomes <MISSING>.,No denominator.,False,False,PASS,True,NaN
3,occupancy_lien_status_group,owner_occupancy_name + ' | ' + lien_status_name,owner_occupancy_name|lien_status_name,Combines two strong loan-structure signals.,categorical,Each missing part becomes <MISSING>.,No denominator.,False,False,PASS,True,NaN
4,loan_type_lien_status_group,loan_type_name + ' | ' + lien_status_name,loan_type_name|lien_status_name,Represents financing type and lien position to...,categorical,Each missing part becomes <MISSING>.,No denominator.,False,False,PASS,True,NaN
5,state_lien_status_group,state_name + ' | ' + lien_status_name,state_name|lien_status_name,Combines two high-importance non-sensitive fie...,categorical,Each missing part becomes <MISSING>.,No denominator.,False,False,PASS,True,NaN
6,property_purpose_group,property_type_name + ' | ' + loan_purpose_name,property_type_name|loan_purpose_name,Adds a compact property and loan-use interaction.,categorical,Each missing part becomes <MISSING>.,No denominator.,False,False,PASS,True,NaN
7,agency_lien_status_group,agency_name + ' | ' + lien_status_name,agency_name|lien_status_name,Adds a compact agency and lien interaction.,categorical,Each missing part becomes <MISSING>.,No denominator.,False,False,PASS,True,NaN
8,applicant_income_to_tract_income,applicant_income_to_area_income / tract_income...,applicant_income_to_area_income|tract_income_r...,Could compare applicant and estimated tract in...,numeric,A missing source gives NaN.,A non-positive denominator gives NaN.,False,False,PASS_BUT_DUPLICATE,False,This is an exact Stage 3 proposal and selected...
9,family_owner_unit_count_difference,number_of_1_to_4_family_units - number_of_owne...,number_of_1_to_4_family_units|number_of_owner_...,Could describe non-owner small-family housing ...,numeric,A missing source gives NaN.,No denominator.,False,False,PASS_BUT_DUPLICATE,False,This is an exact Stage 3 proposal and selected...


## 23. Common Base Pack

`boosting_base_v1` keeps reviewed original and useful Stage 1 fields. It removes the target, IDs used for splitting, sensitive fields, and exact tree duplicates.

In [6]:
boosting_packs = json.loads((ROOT / "artifacts/features/stage4/boosting_feature_packs.json").read_text(encoding="utf-8"))
base_pack = boosting_packs["packs"]["boosting_base_v1"]
assert "loan_amount_000s" not in base_pack["raw"]
display(base_pack)

{'numeric': ['applicant_income_000s',
  'population',
  'hud_median_family_income',
  'number_of_owner_occupied_units',
  'number_of_1_to_4_family_units',
  'applicant_income_to_area_income',
  'tract_income_ratio',
  'owner_occupied_unit_ratio',
  'family_units_per_1000_people',
  'owner_occupied_units_per_1000_people',
  'has_co_applicant'],
 'categorical': ['agency_name',
  'loan_type_name',
  'property_type_name',
  'loan_purpose_name',
  'owner_occupancy_name',
  'preapproval_name',
  'state_name',
  'lien_status_name',
  'loan_program_group',
  'applicant_income_area_group',
  'tract_income_level',
  'us_region'],
 'raw': ['applicant_income_000s',
  'population',
  'hud_median_family_income',
  'number_of_owner_occupied_units',
  'number_of_1_to_4_family_units',
  'applicant_income_to_area_income',
  'tract_income_ratio',
  'owner_occupied_unit_ratio',
  'family_units_per_1000_people',
  'owner_occupied_units_per_1000_people',
  'has_co_applicant',
  'agency_name',
  'loan_type_n

## 24. Initial Engineered Pack

`boosting_engineered_v1` adds two numeric fields and six small category combinations. These are fixed row-level calculations and use no learned statistics.

In [7]:
engineered_pack = boosting_packs["packs"]["boosting_engineered_v1"]
assert len(engineered_pack["fixed_features"]) == 8
display(engineered_pack)

{'numeric': ['applicant_income_000s',
  'population',
  'hud_median_family_income',
  'number_of_owner_occupied_units',
  'number_of_1_to_4_family_units',
  'applicant_income_to_area_income',
  'tract_income_ratio',
  'owner_occupied_unit_ratio',
  'family_units_per_1000_people',
  'owner_occupied_units_per_1000_people',
  'has_co_applicant',
  'estimated_tract_family_income_000s',
  'applicant_vs_area_income_gap_000s'],
 'categorical': ['agency_name',
  'loan_type_name',
  'property_type_name',
  'loan_purpose_name',
  'owner_occupancy_name',
  'preapproval_name',
  'state_name',
  'lien_status_name',
  'loan_program_group',
  'applicant_income_area_group',
  'tract_income_level',
  'us_region',
  'purpose_lien_status_group',
  'occupancy_lien_status_group',
  'loan_type_lien_status_group',
  'state_lien_status_group',
  'property_purpose_group',
  'agency_lien_status_group'],
 'raw': ['applicant_income_000s',
  'population',
  'hud_median_family_income',
  'number_of_owner_occupied_u

## 25. CatBoost Native Pack

`catboost_native_v1` keeps lender, metro area, county, tract, state, and other categories as native CatBoost text fields. Rare grouping is learned only inside the Pipeline.

In [8]:
catboost_schema = json.loads((ROOT / "artifacts/features/stage4/catboost_feature_schema.json").read_text(encoding="utf-8"))
assert catboost_schema["categorical_strategy"].startswith("native CatBoost")
display(catboost_schema)

{'stage': 'stage4b',
 'version': 'stage4b_initial_boosting_packs_v1_20260714',
 'feature_pack': 'catboost_native_v1',
 'raw_input_columns': ['applicant_income_000s',
  'population',
  'hud_median_family_income',
  'number_of_owner_occupied_units',
  'number_of_1_to_4_family_units',
  'applicant_income_to_area_income',
  'tract_income_ratio',
  'owner_occupied_unit_ratio',
  'family_units_per_1000_people',
  'owner_occupied_units_per_1000_people',
  'has_co_applicant',
  'agency_name',
  'loan_type_name',
  'property_type_name',
  'loan_purpose_name',
  'owner_occupancy_name',
  'preapproval_name',
  'state_name',
  'lien_status_name',
  'loan_program_group',
  'applicant_income_area_group',
  'tract_income_level',
  'us_region',
  'respondent_id',
  'msamd_name',
  'county_name',
  'census_tract_number'],
 'numeric_features': ['applicant_income_000s',
  'population',
  'hud_median_family_income',
  'number_of_owner_occupied_units',
  'number_of_1_to_4_family_units',
  'applicant_income

## 26. LightGBM Pack

`lightgbm_encoded_v1` uses fold-fit frequency values for high-cardinality fields and fold-fit ordinal values for the remaining categories. Both steps stay inside the Pipeline.

In [9]:
lightgbm_schema = json.loads((ROOT / "artifacts/features/stage4/lightgbm_feature_schema.json").read_text(encoding="utf-8"))
assert "pipeline" in lightgbm_schema["categorical_strategy"]
display(lightgbm_schema)

{'stage': 'stage4b',
 'version': 'stage4b_initial_boosting_packs_v1_20260714',
 'feature_pack': 'lightgbm_encoded_v1',
 'raw_input_columns': ['applicant_income_000s',
  'population',
  'hud_median_family_income',
  'number_of_owner_occupied_units',
  'number_of_1_to_4_family_units',
  'applicant_income_to_area_income',
  'tract_income_ratio',
  'owner_occupied_unit_ratio',
  'family_units_per_1000_people',
  'owner_occupied_units_per_1000_people',
  'has_co_applicant',
  'agency_name',
  'loan_type_name',
  'property_type_name',
  'loan_purpose_name',
  'owner_occupancy_name',
  'preapproval_name',
  'state_name',
  'lien_status_name',
  'loan_program_group',
  'applicant_income_area_group',
  'tract_income_level',
  'us_region',
  'respondent_id',
  'msamd_name',
  'county_name',
  'census_tract_number'],
 'numeric_features': ['applicant_income_000s',
  'population',
  'hud_median_family_income',
  'number_of_owner_occupied_units',
  'number_of_1_to_4_family_units',
  'applicant_incom

## 27. XGBoost Pack

`xgboost_sparse_v1` uses sparse one-hot output for controlled categories. High-cardinality fields use fold-fit frequency values, so the matrix does not become uncontrolled and dense.

In [10]:
xgboost_schema = json.loads((ROOT / "artifacts/features/stage4/xgboost_feature_schema.json").read_text(encoding="utf-8"))
assert xgboost_schema["sparse_output"] is True
display(xgboost_schema)

{'stage': 'stage4b',
 'version': 'stage4b_initial_boosting_packs_v1_20260714',
 'feature_pack': 'xgboost_sparse_v1',
 'raw_input_columns': ['applicant_income_000s',
  'population',
  'hud_median_family_income',
  'number_of_owner_occupied_units',
  'number_of_1_to_4_family_units',
  'applicant_income_to_area_income',
  'tract_income_ratio',
  'owner_occupied_unit_ratio',
  'family_units_per_1000_people',
  'owner_occupied_units_per_1000_people',
  'has_co_applicant',
  'agency_name',
  'loan_type_name',
  'property_type_name',
  'loan_purpose_name',
  'owner_occupancy_name',
  'preapproval_name',
  'state_name',
  'lien_status_name',
  'loan_program_group',
  'applicant_income_area_group',
  'tract_income_level',
  'us_region',
  'respondent_id',
  'msamd_name',
  'county_name',
  'census_tract_number'],
 'numeric_features': ['applicant_income_000s',
  'population',
  'hud_median_family_income',
  'number_of_owner_occupied_units',
  'number_of_1_to_4_family_units',
  'applicant_income_

## 28. Serializable Feature Transformers

Five named transformers accept pandas DataFrames, keep row order, avoid source mutation, handle unseen categories, and reload in a clean process.

In [11]:
transformer_roundtrips = pd.read_csv(ROOT / "artifacts/features/stage4/transformer_roundtrip_results.csv")
assert transformer_roundtrips["status"].eq("PASS").all()
display(transformer_roundtrips)

,transformer,fit_rows,output_rows,output_columns,row_order_preserved,source_unchanged,reload_equal,clean_process_import,unseen_category_handled,status,source_training_sample_unchanged
0,Stage4FixedFeatureEngineer,500,500,43,True,True,True,True,True,PASS,True
1,Stage4CategoricalSanitizer,500,500,43,True,True,True,True,True,PASS,True
2,Stage4RareCategoryGrouper,500,500,43,True,True,True,True,True,PASS,True
3,Stage4FrequencyEncoder,500,500,47,True,True,True,True,True,PASS,True
4,Stage4ColumnSelector,500,500,23,True,True,True,True,True,PASS,True


## 29. Compatibility Smoke Tests

Each available package fits only five trees or iterations on 4,000 saved Train rows and predicts 1,000 other saved Train rows. These tests check compatibility only. They are not model screening.

In [12]:
smoke_tests = json.loads((ROOT / "artifacts/reports/stage4b_smoke_tests.json").read_text(encoding="utf-8"))
assert smoke_tests["status"] == "PASS"
smoke_table = pd.DataFrame(smoke_tests["results"]).T.reset_index(drop=True)
display(smoke_table[["model", "fit_rows", "validation_rows", "fit_seconds", "finite_predictions", "feature_names_stable", "serialization_reload_match", "representation_sparse", "status"]])

,model,fit_rows,validation_rows,fit_seconds,finite_predictions,feature_names_stable,serialization_reload_match,representation_sparse,status
0,catboost,4000,1000,0.358762,True,True,True,False,PASS
1,lightgbm,4000,1000,0.090841,True,True,True,False,PASS
2,xgboost,4000,1000,0.117423,True,True,True,True,PASS


## 30. Stage 4B Artifact Summary

The Stage 4B files record the audit, proposals, five Feature Packs, three schemas, transformer checks, leakage review, and bounded smoke evidence.

In [13]:
artifact_summary = json.loads((ROOT / "artifacts/manifests/stage4/stage4b_artifact_summary.json").read_text(encoding="utf-8"))
display(artifact_summary)

{'stage': 'stage4b',
 'version': 'stage4b_initial_boosting_packs_v1_20260714',
 'created_at_utc': '2026-07-14T09:52:01.999722+00:00',
 'feature_audit_rows': 35,
 'proposal_rows': 12,
 'selected_feature_rows': 8,
 'rejected_feature_rows': 4,
 'feature_packs': ['boosting_base_v1',
  'boosting_engineered_v1',
  'catboost_native_v1',
  'lightgbm_encoded_v1',
  'xgboost_sparse_v1'],
 'schemas': ['catboost', 'lightgbm', 'xgboost'],
 'transformer_roundtrip_rows': 5,
 'leakage_review_status': 'PASS',
 'smoke_status': 'PASS',
 'internal_verification_status': 'PASS',
 'test_rows_used': 0,
 'real_boosting_screening_performed': False,
 'status': 'PASS'}

## 31. Stage 4B Verification

Internal verification checks leakage, pack design, serialization, smoke limits, Test exclusion, artifact completeness, and protected hashes. Final completion also needs two matching notebook runs and independent review.

In [14]:
stage4b_internal = b.build_internal_verification(ROOT)
assert stage4b_internal["status"] == "PASS"
display(pd.DataFrame([stage4b_internal["checks"]]).T.rename(columns={0: "passed"}))

,passed
starting_requirements_pass,True
proposal_limit_met,True
selected_features_exist,True
selected_features_target_independent,True
selected_features_not_sensitive_derived,True
no_banned_feature_in_packs,True
five_feature_packs_exist,True
native_catboost_categories_preserved,True
xgboost_sparse_design,True
learned_steps_pipeline_bound,True


## 32. Stage 4B Completion Note

The initial Feature Packs are ready for the first CatBoost experiment after external execution and review checks pass. No real boosting comparison or tuning was performed.

In [15]:
completion_note = {
    "stage": "Stage 4B — Initial Boosting Feature Packs",
    "implementation_status": "PASS",
    "feature_packs": list(boosting_packs["packs"]),
    "selected_fixed_features": int(initial_feature_proposals["selected"].sum()),
    "test_rows_used": 0,
    "real_screening_performed": False,
    "next_step": "Begin Stage 4C — Initial CatBoost Model and Importance Analysis.",
}
display(completion_note)

{'stage': 'Stage 4B — Initial Boosting Feature Packs',
 'implementation_status': 'PASS',
 'feature_packs': ['boosting_base_v1',
  'boosting_engineered_v1',
  'catboost_native_v1',
  'lightgbm_encoded_v1',
  'xgboost_sparse_v1'],
 'selected_fixed_features': 8,
 'test_rows_used': 0,
 'real_screening_performed': False,
 'next_step': 'Begin Stage 4C — Initial CatBoost Model and Importance Analysis.'}